# 02 — Fetch current annotations and build the crosswalk

Resolve Liu's 2014 locus tags against current annotations without dropping source rows. Direct AO090 matching is evaluated separately from metadata fan-out so multiple accessions or KO assignments do not create false unresolved genes.

In [1]:
import json, os, sys
from datetime import date
from pathlib import Path

import pandas as pd
import yaml

sys.path.insert(0, os.path.abspath(".."))
from atlas import crosswalk, fetch
from atlas.schema import RecordOrigin, RecordType

with open("../config/sources.yaml", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)
Path("../data/raw").mkdir(parents=True, exist_ok=True)
Path("../data/interim").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

## Verify KEGG organism and locus-tag assumption

In [2]:
kegg_cfg = cfg["sources"]["kegg"]
client = fetch.CachedClient(
    cache_dir="../data/raw",
    rate_limit_seconds=kegg_cfg["rate_limit_seconds"],
)
base = kegg_cfg["base_url"]
org = cfg["organism"]["kegg_org"]
kegg_available = False
kegg_error = None
try:
    kegg_genes = fetch.kegg_gene_list(client, base, org)
    kegg_available = not kegg_genes.empty
    if not kegg_available:
        raise RuntimeError(f"KEGG returned no genes for organism: {org}")
    pattern_rate = kegg_genes["kegg_gene_id"].str.fullmatch(r"aor:AO090\d{9}").mean()
    assert pattern_rate > 0.8, "KEGG IDs no longer support direct AO090 matching"
    print("KEGG genes:", len(kegg_genes), "| exact AO090 format:", f"{pattern_rate:.1%}")
except Exception as exc:
    kegg_error = f"{type(exc).__name__}: {exc}"
    print("KEGG unavailable; continuing with UniProt-backed identifiers:", kegg_error)
    kegg_genes = pd.DataFrame(columns=["kegg_gene_id", "kegg_description"])

KEGG genes: 12102 | exact AO090 format: 99.8%


## Retrieve KEGG cross-references, KO links, and pathway membership

In [3]:
if kegg_available:
    kegg_uniprot = fetch.kegg_conv(client, base, org, "uniprot")
    kegg_ncbi = fetch.kegg_conv(client, base, org, "ncbi-geneid")
    kegg_ko_df = fetch.kegg_ko(client, base, org)
    pathways = pd.concat(
        [fetch.kegg_pathway_members(client, base, org, ids, group)
         for group, ids in kegg_cfg["pathways"].items()],
        ignore_index=True,
    )
else:
    kegg_uniprot = pd.DataFrame(columns=["kegg_gene_id", "uniprot_accession"])
    kegg_ncbi = pd.DataFrame(columns=["kegg_gene_id", "ncbi_gene_id"])
    kegg_ko_df = pd.DataFrame(columns=["kegg_gene_id", "kegg_ko"])
    pathways = pd.DataFrame(columns=["kegg_gene_id", "kegg_pathway", "pathway_group"])
for label, frame in {
    "genes": kegg_genes, "uniprot links": kegg_uniprot,
    "NCBI links": kegg_ncbi, "KO links": kegg_ko_df,
    "pathway memberships": pathways,
}.items():
    print(f"{label:22s} {len(frame):6d}")

genes                   12102
uniprot links           11624
NCBI links              12102
KO links                 4472
pathway memberships       196


## Retrieve the complete UniProt taxonomy query

In [4]:
up_cfg = cfg["sources"]["uniprot"]
uniprot_df = fetch.normalise_uniprot(fetch.uniprot_proteome(
    client, up_cfg["base_url"], up_cfg["fields"],
    cfg["organism"]["taxon_id"], cfg["organism"].get("uniprot_proteome"),
))
assert uniprot_df["uniprot_accession"].is_unique
assert len(uniprot_df) > 500, "UniProt response appears truncated to one search page"
print("UniProt entries:", len(uniprot_df))
uniprot_df.to_parquet("../data/interim/uniprot_annotations.parquet", index=False)

# Build an atomic locus-to-accession table from explicit AO090 strings.
# NCBI IDs are assigned only when their cardinality aligns with locus tags;
# otherwise the raw value is retained in UniProt annotations for review.
locus_rows = []
for row in uniprot_df.itertuples(index=False):
    gene_text = str(getattr(row, "gene_synonyms_raw", "") or "")
    kegg_text = str(getattr(row, "kegg_gene_id_raw", "") or "")
    tags = list(dict.fromkeys(__import__("re").findall(r"AO090\d{9}", gene_text + " " + kegg_text)))
    ncbi_ids = __import__("re").findall(r"\d+", str(getattr(row, "ncbi_gene_id", "") or ""))
    for position, tag in enumerate(tags):
        ncbi_id = ncbi_ids[position] if len(ncbi_ids) == len(tags) else None
        locus_rows.append({
            "ao_locus_tag": tag,
            "kegg_gene_id": f"aor:{tag}" if f"aor:{tag}" in kegg_text else None,
            "uniprot_accession": row.uniprot_accession,
            "ncbi_gene_id": ncbi_id,
            "gene_name": getattr(row, "gene_name", None),
            "function": getattr(row, "function", None),
            "compartment_raw": getattr(row, "compartment_raw", None),
        })
uniprot_locus = pd.DataFrame(locus_rows).drop_duplicates()
print("UniProt AO090 locus rows:", len(uniprot_locus),
      "| unique loci:", uniprot_locus.ao_locus_tag.nunique())
uniprot_locus.to_parquet("../data/interim/uniprot_locus_crosswalk.parquet", index=False)

No proteome pinned; querying by taxonomy (noisier).


UniProt entries: 12050


UniProt AO090 locus rows: 12069 | unique loci: 12064


## Build metadata crosswalk and resolve Liu rows by exact locus tag

In [5]:
if kegg_available:
    cw = crosswalk.build_crosswalk(kegg_genes, kegg_uniprot, kegg_ncbi, kegg_ko_df)
    identity_lookup = kegg_genes[["kegg_gene_id"]].copy()
    identity_lookup["ao_locus_tag"] = identity_lookup["kegg_gene_id"].str.split(":").str[-1]
    identity_source = "KEGG"
else:
    cw = uniprot_locus.copy()
    identity_lookup = uniprot_locus[["ao_locus_tag", "kegg_gene_id"]].drop_duplicates("ao_locus_tag")
    identity_source = "UniProt AO090 cross-reference"
cw.to_parquet("../data/interim/crosswalk.parquet", index=False)
pathways.to_parquet("../data/interim/kegg_pathways.parquet", index=False)

# Resolve identity against one row per KEGG gene. Metadata can be one-to-many
# and must not turn an exact biological locus match into false ambiguity.
assert identity_lookup["ao_locus_tag"].is_unique

seed = pd.read_csv("../data/interim/liu_components_raw_cleaned.csv")
seed = seed.rename(columns={"ID": "liu_ao_locus_tag"})
assert len(seed) == 369 and seed["liu_ao_locus_tag"].notna().all()
seed = crosswalk.make_record_ids(seed)
resolved = crosswalk.resolve_by_locus_tag(seed, identity_lookup)
crosswalk.assert_no_row_loss(seed, resolved, "record_id")
assert resolved["record_id"].is_unique
resolved["record_origin"] = RecordOrigin.LIU2014.value
resolved["record_type"] = RecordType.MACHINERY.value
print(resolved["mapping_status"].value_counts(dropna=False))

Crosswalk fan-out: 12102 -> 12103 rows. A gene maps to multiple accessions; this is expected but must be handled explicitly.


mapping_status
exact         368
unresolved      1
Name: count, dtype: int64


In [6]:
unresolved = crosswalk.report_unresolved(resolved)
unresolved.to_csv("../data/processed/unresolved_identifiers.csv", index=False)
resolved.to_parquet("../data/interim/seed_resolved.parquet", index=False)

report = {
    "retrieved_on": date.today().isoformat(),
    "kegg_available": kegg_available,
    "kegg_error": kegg_error,
    "identity_source": identity_source,
    "kegg_gene_rows": len(kegg_genes),
    "uniprot_rows": len(uniprot_df),
    "crosswalk_rows": len(cw),
    "liu_rows": len(seed),
    "exact_rows": int((resolved.mapping_status == "exact").sum()),
    "flagged_rows": len(unresolved),
    "kegg_ko_links": len(kegg_ko_df),
    "machinery_with_ko": int(seed.liu_ao_locus_tag.isin(kegg_ko_df.kegg_gene_id.str.split(":").str[-1]).sum()),
}
with open("../data/interim/crosswalk_profile.json", "w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)
print(json.dumps(report, indent=2))

{
  "retrieved_on": "2026-07-29",
  "kegg_available": true,
  "kegg_error": null,
  "identity_source": "KEGG",
  "kegg_gene_rows": 12102,
  "uniprot_rows": 12050,
  "crosswalk_rows": 12103,
  "liu_rows": 369,
  "exact_rows": 368,
  "flagged_rows": 1,
  "kegg_ko_links": 4472,
  "machinery_with_ko": 330
}


## Interpretation

If direct matching leaves unresolved rows, inspect them before implementing cross-reference, sequence, or new-orthology resolution. New orthology must remain distinguishable from Liu's assignments.